

$$ max_{virtuais} = (n-1)\cdot d \cdot Q^{,d-1} $$


onde

* (n) = número de amostras reais,
* (d) = número de entradas (dimensões),
* (Q) = número de quantis (Q_a) usados (isto é, o número de valores de (a) que você fixa para as outras dimensões). 

Aplicando aos seus valores:

* (n=25) (25 amostras) → (n-1=24)
* (d=10) (10 entradas)
* se usar o conjunto sugerido ({0.025,0.25,0.5,0.75,0.975}) → (Q=5)


$$ max_{virtuais} = 24 \times 10 \times 5^{9} = 468,750,000 $$


Ou seja, **468 750 000** amostras virtuais no máximo (explosão combinatória enorme).

Observações práticas importantes

* Esse número é teórico — na prática é inviável computacionalmente e muitas dessas combinações podem ser irrelevantes ou redundantes.
* Os autores mencionam justamente essa explosão e, em aplicações reais, costumam reduzir (Q) (por exemplo, usar somente (a=0.5), i.e. (Q=1)) para diminuir o custo. Com (Q=1) o mesmo cálculo dá (24\times 10 \times 1^{9}=240) (ordem de centenas), muito mais manejável — e coincide com a motivação do artigo para escolher menos quantis em problemas reais. 
* Outro ponto: o número de saídas (1 saída no seu caso) **não** entra na fórmula — só importam entradas/dimensões, quantis e (n).


In [435]:
import numpy as np
import pandas as pd
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, Matern, RationalQuadratic, ExpSineSquared, DotProduct, WhiteKernel, ConstantKernel as C
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
out_scaler = StandardScaler()

In [436]:

PREDICTORS = ["pH", "Cond", "Temp", "OD", "Tds", "Resist", "Salin", "ORP", "IP", "Cor"] # 10 entradas
PREDICTORS = ["pH", "Temp", "IP"] # 10 entradas

TARGETS = ["Fe", "Al", "As", "Pb", "Zn", "Hg", "Co", "V", "Ba", "Mn"] # 10 saidas

In [437]:
Data = pd.read_excel("./Dados/Dados.xlsx")

Datasets = []

for n in range(1, 5):
    n_data = Data[Data["Pontos"] == f"P{n}" ]
    n_data = n_data.drop(columns=["Campanhas", "Pontos"])

    Datasets.append(n_data)

    

# Métricas

In [438]:
import matplotlib.pyplot as plt
import numpy as np
import os

def PlotPredictions(train_orig, train_pred, test_orig, test_pred, target_name, n): 
    # cria figura
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # -----------------------------
    # SUBPLOT 1 — TREINO
    # -----------------------------
    ax = axes[0]
    n_train = len(train_orig)
    x_train = np.arange(n_train)

    ax.plot(x_train, train_orig, label="Original (train)", color="blue", )
    ax.plot(x_train, train_pred, label="Predito (train)", color="red",)

    ax.scatter(x_train, train_orig, color="blue", s=35)
    ax.scatter(x_train, train_pred, color="red", s=35)

    ax.set_title("Treinamento")
    ax.set_xlabel("Amostras")
    ax.set_ylabel(target_name)
    ax.grid(True)
    ax.legend()

    # -----------------------------
    # SUBPLOT 2 — TESTE
    # -----------------------------
    ax = axes[1]
    n_test = len(test_orig)
    x_test = np.arange(n_test)

    ax.plot(x_test, test_orig, label="Original (test)", color="blue", linewidth=1.8)
    ax.plot(x_test, test_pred, label="Predito (test)", color="red", linewidth=1.8)

    ax.scatter(x_test, test_orig, color="blue", s=35)
    ax.scatter(x_test, test_pred, color="red", s=35)

    ax.set_title("Teste")
    ax.set_xlabel("Amostras")
    ax.set_ylabel(target_name)
    ax.grid(True)
    ax.legend()

    plt.tight_layout()

    # salva a figura
    filename = f"./Dados/VirtualData/P{n}/TrainResults/{target_name}.pdf"
    plt.savefig(filename, format="pdf", bbox_inches="tight")

    # fecha a figura (IMPORTANTE para não acumular memória)
    plt.close(fig)

    print(f"Figura salva em: {filename}")


In [439]:
from sklearn.metrics import r2_score, mean_squared_error 

def ComputeMetrics(model, df_train, df_test, target, n):
    
    test_pred = out_scaler.inverse_transform(model.execute(df_test[PREDICTORS]).reshape(-1, 1))
    train_pred = out_scaler.inverse_transform(model.execute(df_train[PREDICTORS]).reshape(-1, 1))
    
    test_orig = out_scaler.inverse_transform(df_test[[target]])
    train_orig = out_scaler.inverse_transform(df_train[[target]])
    
    PlotPredictions(train_orig, train_pred, test_orig, test_pred, target, n)
    return {
        'r2_test': r2_score(test_orig, test_pred),
        'mse_test': mean_squared_error(test_orig, test_pred),
        'r2_train': r2_score(train_orig, train_pred),
        'mse_train': mean_squared_error(train_orig, train_pred),
    }

# Extraindo dados

In [440]:
Krbf = C(1.0) * RBF(length_scale=1.0, length_scale_bounds=(1e-9, 1e+6)) # R2 < 0
Kmtrn = C(1.0) * Matern(length_scale=1.0, nu=0.5, length_scale_bounds=(1e-9, 1e+6)) # R2 < 0
Krq = C(1.0) * RationalQuadratic(length_scale=1.0, alpha=0.1, length_scale_bounds=(1e-9, 1e+6)) # 
Kess = C(1.0) * ExpSineSquared(length_scale=1.0, periodicity=3.0, length_scale_bounds=(1e-9, 1e+6)) 
Kdp = C(1.0) * DotProduct() + WhiteKernel()

In [441]:

def PrepareData(Dataset, target):
    scaler.fit(Dataset[PREDICTORS])
    out_scaler.fit(Dataset[[target]])
    
    TrainData, TestData = train_test_split(Dataset, test_size=0.2)
    NormTrainData, NormTestData = TrainData, TestData 
    
    NormTrainData[PREDICTORS] = scaler.transform(TrainData[PREDICTORS])
    NormTrainData[target] = out_scaler.transform(TrainData[[target]])
    
    NormTestData[PREDICTORS] = scaler.transform(TestData[PREDICTORS])
    NormTestData[target] = out_scaler.transform(TestData[[target]])
    
    return TrainData, NormTrainData, TestData, NormTestData


In [442]:
kernel =  Kmtrn

class GPRVSG:
    def __init__(self, Dataset, target):
        self.target = target
        self.TrainData, self.NormTrainData, self.TestData, self.NormTestData = PrepareData(Dataset,
                                                                                           target)
        self.x_train =  self.NormTrainData[PREDICTORS].values
        self.y_train = self.NormTrainData[target].values
        
        self.x_test = self.NormTestData[PREDICTORS].values
        self.y_test = self.NormTestData[target].values
        
        self.d = self.x_train.shape[1]
        self.virtual_samples_x = []
        self.virtual_samples_y = []

    def ComputeProjection(self):
        projections = []
        for m in range(self.d):
            x_m = self.x_train[:, m]         # valores reais da dimensão m
            s_m = np.sort(x_m)               # projeções ordenadas
            projections.append(s_m)
        return projections


    def SetInputSpace(self):
        projections = self.ComputeProjection()
        avg_dists = [np.mean(np.diff(proj)) for proj in projections]

        Q_alpha = np.quantile(projections, [0.5], axis=1, method="hazen").T
        for m in range(self.d):
            for i in range(len(projections[m]) - 1):
                dist = projections[m][i + 1] - projections[m][i]
                if dist > avg_dists[m]:
                    G = (projections[m][i] + projections[m][i + 1]) / 2
                    for q in range(self.d):
                        if q != m:
                            for quantile in Q_alpha[q]:
                                tilde_q = np.zeros(self.d)
                                tilde_q[m] = G
                                tilde_q[q] = quantile
                                self.virtual_samples_x.append(tilde_q)
        self.virtual_samples_x = np.array(self.virtual_samples_x)

    def BuildModel(self):
        self.model = GaussianProcessRegressor(kernel=kernel, n_restarts_optimizer=100,
                                              alpha=1e-8, normalize_y=True)
        self.model.fit(self.x_train, self.y_train)

    def getX(self):
        n_dims = self.virtual_samples_x.shape[1]
        col_names = [f"x{i+1}" for i in range(n_dims)]
        df = pd.DataFrame(self.virtual_samples_x, columns=col_names)
        return [df[col] for col in col_names]

    def ComputeY(self,):
        # pega lista de colunas: [x1, x2, ..., xn]
        X_cols = self.getX()

        # monta matriz X de entrada
        X = np.column_stack(X_cols)

        # prediz y
        self.virtual_samples_y = self.model.predict(X)

        # cria nomes x1, x2, ..., xn
        n_dims = len(X_cols)

        # monta DataFrame final
        data = {name: X_cols[i] for i, name in enumerate(PREDICTORS)}
        data[self.target] = self.virtual_samples_y

        self.virtual_samples_df = pd.DataFrame(data)

    def execute(self, *coords):
        # transformar lista de vetores em matriz X
        X = np.column_stack(coords)

        y_pred = self.model.predict(X)
        return y_pred

    def Run(self):
        self.SetInputSpace()
        self.BuildModel()
        self.ComputeY()

In [443]:
def PlotVirtualData(virtual_df, original_df, predictors, target, n):
    savepath = f"./Dados/VirtualData/P{n+1}/VSGResults/Virtual_{target}.pdf"

    # total = 10 entradas + 1 saída
    total_features = len(predictors) + 1
    fig, axes = plt.subplots(total_features, 1, figsize=(10, 2*total_features), sharex=False)

    # junta entradas + saída
    features = predictors + [target]

    for i, feat in enumerate(features):

        ax = axes[i]

        # valores preditos (virtuais)
        y_pred = virtual_df[feat].values
        x_pred = np.arange(len(y_pred))

        # valores originais reais (Dataset)
        y_orig = original_df[feat].values
        x_orig = np.arange(len(y_orig))

        # plot
        ax.plot(x_pred, y_pred, color="red", label="Virtual (predito)", linewidth=1.7)
        ax.scatter(x_pred, y_pred, color="red", s=20)

        ax.plot(x_orig, y_orig, color="blue", label="Original", linewidth=1.7)
        ax.scatter(x_orig, y_orig, color="blue", s=20)

        ax.set_ylabel(feat)
        ax.grid(True)

        if i == 0:
            ax.legend()

    axes[-1].set_xlabel("Amostras")

    plt.tight_layout()
    plt.savefig(savepath, format="pdf", bbox_inches="tight")
    plt.close(fig)

    print(f"Figura salva em: {savepath}")


In [444]:

import os

def GetVirtualData():
    for n in range(0, 4):  
        metrics_all = []
        output_dir = f"./Dados/VirtualData/P{n+1}"
        os.makedirs(output_dir, exist_ok=True)
        os.makedirs(f"./Dados/VirtualData/P{n+1}/TrainResults/", exist_ok=True)
        os.makedirs(f"./Dados/VirtualData/P{n+1}/VSGResults/", exist_ok=True)

        for target in TARGETS:            
            gpr_vsg = GPRVSG(Datasets[n], target)
            gpr_vsg.Run()
            metrics = ComputeMetrics(gpr_vsg,gpr_vsg.NormTrainData,gpr_vsg.NormTestData,target,n+1)

            vs_filename = os.path.join(output_dir, f"virtual_samples_{target}.xlsx")
            gpr_vsg.virtual_samples_df.to_excel(vs_filename, index=False)

            # Guarda métricas + nome do target
            metrics_all.append(metrics)
            metrics["target"] = target  
            
            PlotVirtualData(
                virtual_df = gpr_vsg.virtual_samples_df,          # dados virtuais desnormalizados
                original_df = Datasets[n],                        # dataset original desnormalizado
                predictors = PREDICTORS,                          # entradas
                target = target,                                  # saída
                n = n
            )
            # break
        # Converte lista de dicts para DataFrame
        df_metrics = pd.DataFrame(metrics_all)
        display(df_metrics)

        # Salva arquivo final por ponto
        metrics_filename = os.path.join(output_dir, "GprMetrics.xlsx")
        df_metrics.to_excel(metrics_filename, index=False)
        break

In [445]:
GetVirtualData()

Figura salva em: ./Dados/VirtualData/P1/TrainResults/Fe.pdf
Figura salva em: ./Dados/VirtualData/P1/VSGResults/Virtual_Fe.pdf
Figura salva em: ./Dados/VirtualData/P1/TrainResults/Al.pdf
Figura salva em: ./Dados/VirtualData/P1/VSGResults/Virtual_Al.pdf
Figura salva em: ./Dados/VirtualData/P1/TrainResults/As.pdf
Figura salva em: ./Dados/VirtualData/P1/VSGResults/Virtual_As.pdf
Figura salva em: ./Dados/VirtualData/P1/TrainResults/Pb.pdf
Figura salva em: ./Dados/VirtualData/P1/VSGResults/Virtual_Pb.pdf


d:\Users\sarah\Educ\Ic-2025.2\tf-env\lib\site-packages\sklearn\gaussian_process\kernels.py:440: ConvergenceWarning: The optimal value found for dimension 0 of parameter k2__length_scale is close to the specified lower bound 1e-09. Decreasing the bound and calling fit again may find a better value.
  warnings.warn(


Figura salva em: ./Dados/VirtualData/P1/TrainResults/Zn.pdf
Figura salva em: ./Dados/VirtualData/P1/VSGResults/Virtual_Zn.pdf
Figura salva em: ./Dados/VirtualData/P1/TrainResults/Hg.pdf
Figura salva em: ./Dados/VirtualData/P1/VSGResults/Virtual_Hg.pdf
Figura salva em: ./Dados/VirtualData/P1/TrainResults/Co.pdf
Figura salva em: ./Dados/VirtualData/P1/VSGResults/Virtual_Co.pdf
Figura salva em: ./Dados/VirtualData/P1/TrainResults/V.pdf
Figura salva em: ./Dados/VirtualData/P1/VSGResults/Virtual_V.pdf
Figura salva em: ./Dados/VirtualData/P1/TrainResults/Ba.pdf
Figura salva em: ./Dados/VirtualData/P1/VSGResults/Virtual_Ba.pdf
Figura salva em: ./Dados/VirtualData/P1/TrainResults/Mn.pdf
Figura salva em: ./Dados/VirtualData/P1/VSGResults/Virtual_Mn.pdf


,r2_test,mse_test,r2_train,mse_train,target
0,0.478696,48201.668720,1.0,5.296449e-11,Fe
1,0.023539,12276.757928,1.0,6.956455e-13,Al
2,-0.180060,0.001038,1.0,1.534442e-19,As
3,-0.395776,0.237750,1.0,4.090758e-17,Pb
4,-13.970050,498.376198,1.0,8.053615e-14,Zn
5,-0.564486,0.054472,1.0,6.314911e-19,Hg
6,-1.670893,0.033665,1.0,1.725604e-17,Co
7,0.605460,0.042391,1.0,9.512103e-18,V
8,0.183049,56.585673,1.0,1.521427e-14,Ba
9,-0.088783,3465.411038,1.0,6.653499e-13,Mn
